In [1]:
# 1. Install Kaggle library and upload your API token
!pip install -q kaggle
from google.colab import files
uploaded = files.upload() # Choose your kaggle.json file here

# 2. Move the kaggle.json file to the correct hidden directory
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 3. Download a standard 4-class weather dataset
# (This downloads the dataset directly to Colab's servers)
!kaggle datasets download -d pratik2901/multiclass-weather-dataset
!unzip -q multiclass-weather-dataset -d weather_data/

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/pratik2901/multiclass-weather-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
100% 91.4M/91.4M [00:06<00:00, 13.9MB/s]



In [11]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Define our image size and batch size
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
DATA_DIR = "weather_data/Multi-class Weather Dataset" # Adjust this path if the unzipped folder structure varies

# Load the Training Data (80% of the images)
train_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Load the Validation Data (20% of the images)
val_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Get the class names (e.g., Sunny, Cloudy, Rainy, Snowy, Foggy)
class_names = train_dataset.class_names
print(f"Weather Classes: {class_names}")

# Configure the dataset for performance (caches data in memory so it trains faster)
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.cache().prefetch(buffer_size=AUTOTUNE)

Found 1125 files belonging to 4 classes.
Using 900 files for training.
Found 1125 files belonging to 4 classes.
Using 225 files for validation.
Weather Classes: ['Cloudy', 'Rain', 'Shine', 'Sunrise']


In [9]:
!ls weather_data/

'Multi-class Weather Dataset'


In [13]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

# 1. The Anti-Overfitting Architecture
# Initialize a sequential model (a linear stack of layers)

model = keras.Sequential([
    # AGGRESSIVE AUGMENTATION: Cranked up to 25% to force the model to learn!
    layers.Rescaling(1./255, input_shape=(224, 224, 3)),  # Normalization: scales pixels from 0-255 to 0-1
    layers.RandomFlip("horizontal"),                      # Augmentation
    layers.RandomRotation(0.25),                          # Augmentation
    layers.RandomZoom(0.25),                              # Augmentation

    # Block 1
    #Input & First Convolutional Block
    # Applies 32 different 3x3 filters to the input images
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Dropout(0.2), # DEEP DROPOUT: Turn off 20% of neurons

    # Block 2
    #Second Convolutional Block
    # Increases filters to 64 as the model looks for more complex features
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Dropout(0.3), # DEEP DROPOUT: Turn off 30% of neurons

    # Block 3
    #Third Convolutional Block
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Dropout(0.4), # DEEP DROPOUT: Turn off 40% of neurons

    # Block 4
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(pool_size=(2, 2)),

    # Feature Averaging (keeps parameters safely around ~259k)
    #Averages the features instead of violently flattening them
    layers.GlobalAveragePooling2D(),

    # Final Classifier
    #Dense (Fully Connected) Classifier Layers
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5), # Final 50% Dropout

    #Output Layer (4 neurons for 4 weather classes)
    layers.Dense(4, activation='softmax')
])

# Print a summary of our newly built architecture
model.summary()

# 2. Compile the model
# We use 'Sparse' because our class labels are integers (0 to 3)
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

# 3. Define the Early Stopping callback
early_stopper = EarlyStopping(
    monitor='val_loss',          # The most important metric to watch
    patience=4,                  # Wait for 4 epochs of no improvement before stopping
    restore_best_weights=True    # Automatically rolls back the model to its best state!
)

# 4. Train the model
# We can safely set a higher maximum epoch limit now (e.g., 30)
# because the callback will stop it long before it reaches the end.
print("Training the Final Anti-Overfitting Model... ")
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=30,
    callbacks=[early_stopper]
)

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_2 (Rescaling)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip_2 (RandomFlip)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation_2               │ (None, 224, 224, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom_2 (RandomZoom)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 222, 222, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 109, 109, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 52, 52, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 24, 24, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 24, 24, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 128)            │             

 Total params: 259,268 (1012.77 KB)

 Trainable params: 258,564 (1010.02 KB)

 Non-trainable params: 704 (2.75 KB)

Training the Final Anti-Overfitting Model... 
Epoch 1/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 8s 156ms/step - accuracy: 0.7667 - loss: 0.6515 - val_accuracy: 0.3022 - val_loss: 1.3676
Epoch 2/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 4s 142ms/step - accuracy: 0.8111 - loss: 0.4886 - val_accuracy: 0.2933 - val_loss: 1.9379
Epoch 3/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 4s 141ms/step - accuracy: 0.8733 - loss: 0.3667 - val_accuracy: 0.2756 - val_loss: 3.1369
Epoch 4/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 4s 142ms/step - accuracy: 0.8822 - loss: 0.3388 - val_accuracy: 0.2933 - val_loss: 3.1482
Epoch 5/30
29/29 ━━━━━━━━━━━━━━━━━━━━ 4s 144ms/step - accuracy: 0.8833 - loss: 0.3337 - val_accuracy: 0.3333 - val_loss: 3.0319


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image
from google.colab import files
import matplotlib.pyplot as plt

print("Upload a weather photo to test the model!")
uploaded = files.upload()

for fn in uploaded.keys():
    # 1. Load the image and resize it to the 224x224 shape our model expects
    img = image.load_img(fn, target_size=(224, 224))

    # 2. Display the image
    plt.imshow(img)
    plt.axis('off')
    plt.show()

    # 3. Convert the image to a mathematical array and add a batch dimension
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)

    # 4. Have the model predict!
    predictions = model.predict(img_array)
    predicted_class = class_names[np.argmax(predictions)]
    confidence = np.max(predictions) * 100

    print(f"🌤️ Prediction: {predicted_class}")
    print(f"📊 Confidence: {confidence:.2f}%")